# Multimodal RAG with LangChain and NeMo Retriever

This notebook shows how to ingest a multimodal PDF with the current `nemo_retriever` library API, store the extracted text, table, chart, and image content in local LanceDB, and query it with a LangChain RAG pipeline.

**Note:** This example runs NeMo Retriever locally in-process and writes to a LanceDB directory on disk. You need Python 3.12, the NeMo Retriever Library installed with the local model dependencies, the sample `data/multimodal_test.pdf`, and `NVIDIA_API_KEY` for the hosted NVIDIA LLM used to synthesize the final answer.

Install the framework packages for this notebook and NeMo Retriever from this source checkout.

In [ ]:
%pip install -U llama-index llama-index-llms-nvidia langchain langchain-nvidia-ai-endpoints -e "../nemo_retriever[local]"

Configure the sample PDF, the local LanceDB path, and the VL embedding model. The ingest and query paths must use the same embedding model.

In [ ]:
from pathlib import Path

from nemo_retriever import create_ingestor
from nemo_retriever.model import VL_EMBED_MODEL
from nemo_retriever.retriever import Retriever

pdf_path = Path("../data/multimodal_test.pdf").resolve()
LANCEDB_URI = str(Path("./langchain_multimodal_lancedb").resolve())
TABLE_NAME = "nemo-retriever"
TOP_K = 5

VL_EMBED_MODEL

Run NeMo Retriever in-process to extract multimodal content, create VL embeddings, and upload the embedded chunks to local LanceDB.

In [ ]:
ingestor = (
    create_ingestor(run_mode="inprocess")
    .files([str(pdf_path)])
    .extract()
    .embed(
        model_name=VL_EMBED_MODEL,
        embed_model_name=VL_EMBED_MODEL,
        embed_granularity="page",
        local_ingest_embed_backend="hf",
    )
    .vdb_upload(
        vdb_op="lancedb",
        vdb_kwargs={"uri": LANCEDB_URI, "table_name": TABLE_NAME, "overwrite": True},
    )
)

result_df = ingestor.ingest()
result_df[["text"]].head()

Create a small LangChain retriever adapter around `nemo_retriever.retriever.Retriever`. NeMo Retriever performs the query embedding and LanceDB vector search; LangChain handles prompt formatting and answer generation.

In [ ]:
import json
from typing import Any

from langchain_core.documents import Document
from langchain_core.retrievers import BaseRetriever
from langchain_nvidia_ai_endpoints import ChatNVIDIA
from pydantic import ConfigDict, Field


def metadata_from_hit(hit: dict[str, Any]) -> dict[str, Any]:
    metadata = {k: v for k, v in hit.items() if k != "text"}
    for key in ("metadata", "source"):
        value = metadata.get(key)
        if isinstance(value, str):
            try:
                metadata[key] = json.loads(value)
            except json.JSONDecodeError:
                pass
    return metadata


class NemoRetrieverLangChain(BaseRetriever):
    retriever: Retriever = Field(exclude=True)
    top_k: int = 5

    model_config = ConfigDict(arbitrary_types_allowed=True)

    def _get_relevant_documents(self, query: str, *, run_manager=None) -> list[Document]:
        hits = self.retriever.query(query, top_k=self.top_k)
        documents = []
        for rank, hit in enumerate(hits, start=1):
            metadata = metadata_from_hit(hit)
            metadata["rank"] = rank
            documents.append(Document(page_content=str(hit.get("text", "")), metadata=metadata))
        return documents


nemo_retriever = Retriever(
    top_k=TOP_K,
    embed_kwargs={
        "model_name": VL_EMBED_MODEL,
        "embed_model_name": VL_EMBED_MODEL,
        "local_ingest_embed_backend": "hf",
    },
    vdb_kwargs={"uri": LANCEDB_URI, "table_name": TABLE_NAME},
)
langchain_retriever = NemoRetrieverLangChain(retriever=nemo_retriever, top_k=TOP_K)

In [ ]:
import os

from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import PromptTemplate
from langchain_core.runnables import RunnableLambda, RunnablePassthrough

if not os.environ.get("NVIDIA_API_KEY"):
    raise EnvironmentError("Set NVIDIA_API_KEY to use the NVIDIA-hosted LLM for answer generation.")

llm = ChatNVIDIA(model="nvidia/nemotron-3-super-120b-a12b")

def format_docs(docs: list[Document]) -> str:
    return "\n\n".join(doc.page_content for doc in docs)

template = (
    "You are an assistant for question-answering tasks. "
    "Use the following retrieved context to answer the question. "
    "If you don't know the answer, say that you don't know. "
    "Keep the answer concise.\n\n"
    "Context:\n{context}\n\n"
    "Question: {question}"
)

prompt = PromptTemplate.from_template(template)

rag_chain = (
    {"context": langchain_retriever | RunnableLambda(format_docs), "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

Now ask the PDF questions through the LangChain chain.

In [ ]:
rag_chain.invoke("What is the dog doing and where?")

In [ ]:
# Expected answer: the dog is chasing a squirrel in the front yard.
# Exact wording may vary because the final response is generated by the configured LLM.